# Simplest possible quantization: SmolLM2-135M, INT8, absmax per-tensor

Weight-only post-training quantization. No calibration data, no activation quant, no per-channel scales.

**Runtime:** CPU is fine. Runtime → Change runtime type → CPU (or leave T4 if already on).

**What we measure:** memory before/after, per-layer reconstruction error.

In [ ]:
!pip install -q transformers torch

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-135M", torch_dtype=torch.bfloat16
)
print(f"Loaded. Total params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## The quantization function

Symmetric, per-tensor, absmax scaling. Three lines of real math.

In [ ]:
def absmax_int8(w):
    scale = w.abs().max() / 127.0
    w_q   = (w / scale).round().clamp(-127, 127).to(torch.int8)
    w_hat = w_q.to(w.dtype) * scale
    return w_q, w_hat, scale

## Sweep all weight tensors

Skip 1D params (biases, layernorm scales) — quantizing them buys ~nothing and hurts a lot.

In [ ]:
rows, total_fp, total_q = [], 0, 0

for name, w in model.named_parameters():
    if w.ndim < 2 or "weight" not in name:
        continue
    _, w_hat, _ = absmax_int8(w.detach())
    diff = w - w_hat
    rmse = diff.pow(2).mean().sqrt().item()
    rms  = w.pow(2).mean().sqrt().item()
    rel  = rmse / (rms + 1e-12)

    total_fp += w.numel() * w.element_size()
    total_q  += w.numel()        # 1 byte per int8
    rows.append((name, tuple(w.shape), rmse, rel))

print(f"FP weights:   {total_fp/1e6:7.2f} MB")
print(f"INT8 weights: {total_q /1e6:7.2f} MB")
print(f"Compression:  {total_fp/total_q:.2f}x\n")

print(f"{'Layer':<55}{'Shape':<22}{'RMSE':<12}{'rel':<8}")
print("-" * 97)
for name, shape, rmse, rel in rows:
    print(f"{name:<55}{str(shape):<22}{rmse:.2e}   {rel:.4f}")

## What to look for

- **Compression ≈ 2×** (bf16 → int8). Not 4× because the model loaded in bf16, not fp32.
- **Per-tensor relative RMSE** typically 0.3%–2% for linear layers.
- `q_proj` / `k_proj` / `v_proj` usually clean.
- `down_proj` and the LM head / embed are often the worst — fat-tailed weight distributions, one outlier blows up the absmax scale and crushes everyone else's resolution.

That last failure mode is exactly what GPTQ / AWQ / SmoothQuant exist to address.

**Natural next step:** per-channel scaling (one scale per output row of W instead of one for the whole tensor). Usually drops relative error 3–5× for ~free, same storage class.